# Why Dense Networks Fail on Images

A fully-connected network can score 98% on MNIST, which makes it tempting to conclude that dense
layers handle images just fine. They do not. This notebook builds that case in three steps: first
a dense net that looks like it works, then two experiments proving it never used the image
structure at all, and finally a dataset where the illusion collapses completely. The payoff is the
argument for convolution — not as a trick that happens to work better, but as the fix for a
specific, demonstrable blindness.

## Learning objectives

- Build and train dense Keras classifiers on flattened image data.
- Explain why flattening an image destroys spatial information a `Dense` layer could never use
  anyway.
- Predict and verify the effect of permuting every pixel with one fixed shuffle.
- Show that a small translation produces a large change in the vector a dense network sees.
- Explain why data augmentation brute-forces an invariance that convolution provides for free.
- Compare a large dense network against a small CNN on CIFAR-10 by both accuracy and parameter
  count.

## Background

You should already be comfortable building a `Sequential` Keras model from `Dense` layers,
compiling it with an optimizer and a loss, and reading a confusion matrix and classification
report — all introduced in Unit 1.

Two facts about `Dense` layers carry the whole argument here. First, a dense layer connects
**every** input to **every** neuron, so each input has its own independent weight and the layer
has no representation of which inputs are adjacent. Second, feeding an image to such a layer
requires flattening it: a 28×28 image becomes a length-784 vector, and the 2D layout is gone
before the first weight is ever applied.

Everything below is a consequence of those two facts.

## This notebook covers

1. A dense classifier on 8×8 digits — the version that looks like it works
2. Scaling up to MNIST at 28×28
3. The shuffle experiment: does the network even see an image?
4. The translation experiment: a shift is not "small" to a dense network
5. Training on augmented images — brute-forcing the invariance
6. The real test: CIFAR-10, dense versus convolutional
7. Review

**Prerequisites:** `U1_RealEstate-4_Keras.ipynb` and `U1-6_Classify-6_KerasExperiment.ipynb` for
Keras model building; `U2-1_Images-1_SkimageCV2.ipynb` for images as arrays and convolution.

**Datasets:** scikit-learn `load_digits` (8×8 gray-scale), Keras `mnist` (28×28 gray-scale), and
Keras `cifar10` (32×32 color) — all downloaded by their loaders, nothing to install by hand.

**References:** https://keras.io/api/layers/core_layers/dense/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. A dense classifier on 8×8 digits

We start with the smallest image dataset in scikit-learn: 1,797 handwritten digits at 8×8 pixels.
Flattened, each one is a vector of just 64 numbers — small enough that a modest dense network
trains in seconds.

The goal of this section is only to establish the baseline everyone expects: point a dense network
at images, get a good score. Sections 3 and 4 come back and take that result apart.

### 1.1 Load the data

Images are 8×8 pixels.

In [ ]:
from sklearn.datasets import load_digits

# Load the digits dataset
digits = load_digits()
print( dir(digits) )

In [ ]:
# Pick a random index
idx = np.random.randint(0, len(digits.images))

# Display the image
plt.figure(figsize=(4, 4))
plt.imshow(digits.images[idx], cmap='gray')
plt.title(f"Label: {digits.target[idx]}")
plt.axis('off')
plt.show()

### 1.2 Build, train, and evaluate the model

The architecture is a plain funnel — 64 inputs down through 32, 16, and 8 units to a 10-way
softmax. Nothing about it is image-aware; it would work identically on 64 columns of a
spreadsheet.

Training and evaluation both go through `helpers.train_and_evaluate`, the shared course helper: it
fits the model, plots the learning curve beside a confusion matrix, and prints the classification
report. Every model in this notebook is evaluated the same way, which keeps the comparisons honest.

In [ ]:
X = digits.data
y = digits.target

print(X.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

dropout_rate = 0.1

n_features = X.shape[1]
n_classes  = np.unique(y).shape[0]

# Create model
model = Sequential([
    Input(shape=(n_features,)),
    
    Dense(32),
    #BatchNormalization(),
    Activation('relu'),
    Dropout(dropout_rate),
    
    Dense(16),
    #BatchNormalization(),
    Activation('relu'),
    Dropout(dropout_rate),
    
    Dense(8, activation='relu'),
    
    Dense(n_classes, activation='softmax'),
])

# Define the optimizer with a custom learning rate
optimizer = Adam(learning_rate=0.01)

# Compile model
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=25,          # Stop after 5 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

# Display model summary
model.summary()

In [ ]:
model, history = helpers.train_and_evaluate(
    model,
    X_train, y_train,
    X_test, y_test,
    epochs=30,
    batch_size=64,
    callbacks=[early_stopping]
)

## 2. Scaling up: MNIST at 28×28

The same idea on a bigger, harder dataset — 70,000 handwritten digits at 28×28 pixels. Flattened,
each image is now a vector of 784 numbers rather than 64.

That flattening step is worth pausing on. `X_train.reshape(X_train.shape[0], -1)` is where the
image stops being an image: after it runs, nothing in the array records that pixel 0 sat next to
pixel 1 and directly above pixel 28. The network is handed 784 unlabeled numbers.

### 2.1 Load the data

Images are 28×28 pixels.

In [ ]:
from tensorflow.keras.datasets import mnist

# Load data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Print shapes
print("X_train.shape:", X_train.shape)
print("X_test.shape: ", X_test.shape)

In [ ]:
# Display a random image
idx = np.random.randint(0, len(X_train))
plt.figure(figsize=(4, 4))
plt.imshow(X_train[idx], cmap='gray')
plt.title(f"Label: {y_train[idx]}")
plt.axis('off')
plt.show()

### 2.2 Flatten, build, train, and evaluate

Same architecture as section 1, just wider at the input (784 instead of 64) and with batch
normalization switched on. It reaches roughly 98% — the result that makes dense networks look
perfectly adequate for images.

Hold onto that number. The next two sections show it was earned without the network ever using the
fact that these were pictures.

In [ ]:
# flatten images into vectors
X_train_flat = X_train.reshape(X_train.shape[0],-1)
X_test_flat  = X_test.reshape(X_test.shape[0],-1)

print("X_train_flat.shape:", X_train_flat.shape)
print("X_test_flat.shape: ", X_test_flat.shape)

In [ ]:
dropout_rate = 0.1

n_features = X_train_flat.shape[1]
n_classes  = np.unique(y_train).shape[0]

# Create model
model = Sequential([
    Input(shape=(n_features,)),
    
    Dense(32),
    BatchNormalization(),
    Activation('relu'),
    Dropout(dropout_rate),
    
    Dense(16),
    BatchNormalization(),
    Activation('relu'),
    Dropout(dropout_rate),
    
    Dense(8, activation='relu'),
    
    Dense(n_classes, activation='softmax'),
])

# Define the optimizer with a custom learning rate
optimizer = Adam(learning_rate=0.01)

# Compile model
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=25,          # Stop after 5 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

# Display model summary
model.summary()

In [ ]:
model, history = helpers.train_and_evaluate(
    model,
    X_train_flat, y_train,
    X_test_flat, y_test,
    epochs=30,
    batch_size=64,
    callbacks=[early_stopping]
)

## 3. Does a dense network even see an *image*?

The model above hit ~98% on MNIST, so it is tempting to say it "learned to see digits." It did not
— and here is a clean way to prove it.

A `Dense` layer connects **every** input to **every** neuron. It has no notion of which pixels are
next to which; the moment we called `.reshape(..., -1)` and flattened each 28×28 image into a
length-784 vector, the 2D layout was already gone. The network only ever saw a bag of 784 numbers.

So here is the test: pick **one** random shuffle of the 784 positions and apply that *same* shuffle
to every image, train and test alike. To our eyes the digits turn into confetti. If the network is
genuinely using the spatial arrangement of the pixels, its accuracy should crater. If it is just
doing bookkeeping on 784 independent inputs, it should not care at all.

The shuffle is a **bijection** — no pixel is lost or duplicated, only relocated — which the check
in the next cell confirms.

In [ ]:
# One fixed permutation of the 784 pixel positions, applied to EVERY image (train + test).
rng  = np.random.default_rng(42)
perm = rng.permutation(X_train_flat.shape[1])       # e.g. [391, 12, 700, ...]

X_train_perm = X_train_flat[:, perm]
X_test_perm  = X_test_flat[:,  perm]

# Show a few digits before and after the shuffle (reshape the flat vectors back to 28x28).
sample = [0, 1, 2, 3, 4]
fig, axes = plt.subplots(2, len(sample), figsize=(11, 4.5))
for c, k in enumerate(sample):
    axes[0, c].imshow(X_train_flat[k].reshape(28, 28), cmap='gray')
    axes[0, c].set_title(f"label {y_train[k]}", fontsize=9); axes[0, c].axis('off')
    axes[1, c].imshow(X_train_perm[k].reshape(28, 28), cmap='gray')
    axes[1, c].axis('off')
axes[0, 0].set_ylabel("original",  rotation=0, ha='right', va='center')
axes[1, 0].set_ylabel("shuffled",  rotation=0, ha='right', va='center')
fig.suptitle("Same fixed permutation applied to every image", y=1.02)
plt.tight_layout(); plt.show()

# Confirm the permutation really is a bijection over the 784 positions.
print("np.array_equal(sorted(perm), arange):", np.array_equal(np.sort(perm), np.arange(len(perm))))

### 3.1 Train the *same* dense model on the shuffled pixels

Identical architecture and training recipe as the baseline above — only the input pixel order
differs. We give it a separate name (`model_perm`) so the earlier model is left intact.

In [ ]:
n_features = X_train_perm.shape[1]
n_classes  = np.unique(y_train).shape[0]

model_perm = Sequential([
    Input(shape=(n_features,)),

    Dense(32),
    BatchNormalization(),
    Activation('relu'),
    Dropout(dropout_rate),

    Dense(16),
    BatchNormalization(),
    Activation('relu'),
    Dropout(dropout_rate),

    Dense(8, activation='relu'),

    Dense(n_classes, activation='softmax'),
])

optimizer_perm = Adam(learning_rate=0.01)
model_perm.compile(
    optimizer=optimizer_perm,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping_perm = EarlyStopping(
    monitor='val_loss',
    patience=25,
    restore_best_weights=True
)

model_perm.summary()

In [ ]:
model_perm, history_perm = helpers.train_and_evaluate(
    model_perm,
    X_train_perm, y_train,
    X_test_perm, y_test,
    epochs=30,
    batch_size=64,
    callbacks=[early_stopping_perm]
)

In [ ]:
# Side-by-side accuracy: baseline (normal order) vs the model just trained (shuffled order).
# Both use the SAME test digits; they differ only in pixel ordering + a fresh random init.
acc_plain = (model.predict(X_test_flat, verbose=0).argmax(axis=1) == y_test).mean()
acc_perm  = (model_perm.predict(X_test_perm, verbose=0).argmax(axis=1) == y_test).mean()

print(f"Dense on ORIGINAL pixel order : {acc_plain:.4f}")
print(f"Dense on SHUFFLED pixel order : {acc_perm:.4f}")
print(f"Difference                    : {abs(acc_plain - acc_perm):.4f}  (within run-to-run noise)")

### 3.2 Why they match — and why it matters

The two accuracies are essentially the same. A fixed permutation only **relabels which input slot
each pixel lands in**, and the dense layer can undo that relabeling by learning a correspondingly
permuted set of first-layer weights. The scrambled images carry exactly the same information *to
the network* as the originals; the confetti only looks broken to us.

This is the real content of "**dense fails**." The network never used adjacency, so it had nothing
to lose when we destroyed it — and something that cannot tell a pixel from its neighbor cannot
benefit from the fact that ink strokes are *local* and *translate* across the image. That is also
why the augmented digits in section 5 trip it up.

A **convolutional** network would react to this experiment in the opposite way: it would
**collapse**. A convolution slides a small filter over neighboring pixels, hard-wiring the
assumption that nearby pixels belong together. Shatter that adjacency with a permutation and you
take away the exact prior the CNN is built on. *That gap — permutation kills a CNN but not a dense
net — is the whole reason convolutional networks exist.*

## 4. A shift is not "small" to a dense network

Here is the payoff of the shuffle idea. A **convolutional** network is translation-invariant almost
for free: shift the digit and its feature maps just shift too, and pooling ignores *where* a feature
landed — so a digit nudged a few pixels gives nearly the same answer. Does a dense network get
anything like that?

We take one digit and a **slightly translated** copy (a clean 3-pixel diagonal shift, made with
`ImageDataGenerator`), then look at both the way the network does — as flat vectors, and again
after the shuffle. The question: are an image and its small shift *close* in the representation the
dense net actually operates on?

Two quantities in the output make the answer precise. **Cosine similarity** measures the angle
between the two flattened vectors,

$$ \cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}
   {\lVert \mathbf{a} \rVert \, \lVert \mathbf{b} \rVert} $$

and **intersection-over-union** measures how much of the ink actually overlaps. Watch what the
permutation does to the cosine — a permutation is an orthogonal transformation, so it preserves
every dot product and every norm, and the similarity comes out *identical*. The shuffled world and
the natural world are the same world to a dense network.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# One test digit, and a deterministic 3px diagonal translation of it.
ex_idx  = int(np.random.default_rng(7).integers(len(X_test)))
ex_orig = X_test[ex_idx].astype('float32')

datagen_shift = ImageDataGenerator(fill_mode='constant', cval=0.0)
ex_shift = datagen_shift.apply_transform(ex_orig[..., None], {'tx': 3, 'ty': 3})[..., 0]

flat_orig, flat_shift = ex_orig.flatten(), ex_shift.flatten()

# Apply the SAME shuffle used earlier to both.  Note: |perm(a) - perm(b)| == perm(|a - b|),
# so the shuffled "difference" is just the natural difference with its pixels scattered.
p_orig, p_shift = flat_orig[perm], flat_shift[perm]

# Cosine similarity between two flattened images, and the overlap of their ink.
def cos(a, b): return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
ink_o, ink_s = flat_orig > 50, flat_shift > 50
iou = (ink_o & ink_s).sum() / ((ink_o | ink_s).sum() + 1e-9)

panels = [
    ("original",          flat_orig),  ("shifted 3px",       flat_shift),
    ("|difference|",      np.abs(flat_orig - flat_shift)),
    ("original, shuffled", p_orig),     ("shifted, shuffled",  p_shift),
    ("|difference|, shuffled", np.abs(p_orig - p_shift)),
]
fig, axes = plt.subplots(2, 3, figsize=(10, 6.6))
for ax, (title, vec) in zip(axes.ravel(), panels):
    ax.imshow(vec.reshape(28, 28), cmap='gray'); ax.set_title(title, fontsize=10); ax.axis('off')
axes[0, 0].set_ylabel("what we see",    rotation=90, labelpad=12); axes[0, 0].axis('on')
axes[1, 0].set_ylabel("what the net sees", rotation=90, labelpad=12); axes[1, 0].axis('on')
for a in (axes[0, 0], axes[1, 0]): a.set_xticks([]); a.set_yticks([])
plt.suptitle(f"Digit {y_test[ex_idx]} vs. a 3-pixel shift of itself", y=1.0)
plt.tight_layout(); plt.show()

print(f"Cosine similarity, original vs shifted (natural order) : {cos(flat_orig, flat_shift):.3f}")
print(f"Cosine similarity, original vs shifted (shuffled)      : {cos(p_orig,   p_shift):.3f}")
print(f"Overlap of the ink (intersection-over-union)           : {iou:.3f}")

### 4.1 Read the two rows

The two look almost the same to us, yet as vectors they overlap surprisingly little — and the
shuffle leaves that number untouched, because a permutation preserves every dot product.

**Top row (what we see):** the shift is obviously tiny, and the pixels that actually changed
(`|difference|`) form a thin, coherent band along the edges of the strokes — a small, *local*
pattern. This is exactly what a CNN is built to absorb: local filters detect the strokes wherever
they are, and pooling makes the output barely care that they moved. That is where translational
invariance comes from — the *architecture* assumes locality.

**Bottom row (what the dense net sees):** apply the shuffle and that same difference is scattered
into unrelated specks all over the frame. There is no local, coherent change left for any
invariance mechanism to latch onto. And this shuffled world *is* the dense net's world — section 3
proved it cannot tell a pixel from its neighbor, so "shift by three pixels" is not a small,
structured operation to it. It is just a different, largely non-overlapping input vector.

That is the whole point: a dense network gets **no** translational invariance for free. To
recognize the shifted digit it would have to have *learned that specific shift* — every digit in
every position, separately. Which is precisely why the augmented image below trips it up.

### 4.2 Predict an augmented image

Now the consequence, in the plainest possible form. We take a test digit the trained model handles
correctly, apply a random augmentation to it — a rotation, shift, zoom, and shear well within what
a human would call "the same digit" — and ask the model again.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define the image generator with augmentation options
datagen = ImageDataGenerator(
    rotation_range=30,      # Rotate images randomly
    width_shift_range=0.2,  # Randomly shift the width of images
    height_shift_range=0.2, # Randomly shift the height of images
    zoom_range=0.2,         # Randomly zoom
    shear_range=0.1         # Apply shear transformation
)

# Pick a random test image
idx = np.random.randint(0, len(X_test))
image_orig = X_test[idx]

# Create the iterator
iterator = datagen.flow(
    image_orig[np.newaxis, :, :, np.newaxis],
    batch_size=1,
    shuffle=False
)

# Get the augmented image using Python's built-in next()
image_aug = next(iterator)[0, :, :, 0]

# Plot original and augmented image
plt.figure(figsize=(6, 3))

plt.subplot(1, 2, 1)
plt.imshow(image_orig, cmap='gray')
plt.title("Original")

plt.subplot(1, 2, 2)
plt.imshow(image_aug, cmap='gray')
plt.title("Augmented")

plt.tight_layout()
plt.show()

# Ensure image shape is (1, 28, 28, 1) and type is float32
image_orig_input = image_orig.flatten()[np.newaxis, :]
image_aug_input  = image_aug.flatten()[np.newaxis, :]

# Predict using the model
pred_orig = model.predict(image_orig_input)
pred_aug = model.predict(image_aug_input)

# Get predicted digit (argmax of class probabilities)
digit_orig = np.argmax(pred_orig)
digit_aug = np.argmax(pred_aug)

print(f"Original image prediction:  {digit_orig}")
print(f"Augmented image prediction: {digit_aug}")

### 4.3 Why the outer pixels never learn anything

Averaging every training image reveals the problem directly. Only the center pixels ever contain
ink, so the weights attached to the outer pixels receive nothing but zeros throughout training —
they never get a gradient signal worth the name.

A digit that is shifted, scaled, or rotated puts ink exactly where those untrained weights live,
and the network has no idea what to do with it. This is the same story sections 3 and 4.1 told,
now visible in one picture.

In [ ]:
# Display a random image
plt.figure(figsize=(4, 4))
plt.imshow(X_train.mean(axis=0), cmap='gray')
plt.title(f"Avg of all images")
#plt.axis('off')
plt.show()

## 5. Training on augmented images — brute-forcing the invariance

If the network fails on shifted and rotated digits because it never saw any, the obvious fix is to
show it some. **Data augmentation** applies random label-preserving transformations — rotations,
shifts, zooms, shears — to the training images, so the network is forced to handle the variation
explicitly.

It works. The interesting question is what "works" cost us, which section 5.3 takes up.

### 5.1 Create the augmented dataset

We reload MNIST fresh (the earlier cells overwrote `X_train`), then push both the training and test
sets through the generator once to produce permanently augmented copies.

In [ ]:
from tensorflow.keras.datasets import mnist

# Load data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Print shapes
print("X_train.shape:", X_train.shape)
print("X_test.shape: ", X_test.shape)

In [ ]:
# Define the image generator with augmentation options
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.1
)

# Plot the first 3 images before augmentation
plt.figure(figsize=(10, 3))
for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"Original {i}")
    plt.axis('off')
plt.tight_layout()
plt.show()

# Augment X_train
iterator_train = datagen.flow(
    X_train[..., np.newaxis],
    batch_size=X_train.shape[0],
    shuffle=False
)
X_train = next(iterator_train)[..., 0]

# Plot the first 3 images after augmentation
plt.figure(figsize=(10, 3))
for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"Augmented {i}")
    plt.axis('off')
plt.tight_layout()
plt.show()

# Augment X_test
iterator_test = datagen.flow(
    X_test[..., np.newaxis],
    batch_size=X_test.shape[0],
    shuffle=False
)
X_test = next(iterator_test)[..., 0]

### 5.2 Build, train, and evaluate on the augmented data

Same architecture again, trained from scratch on the augmented images. Compare its confusion matrix
to section 2.2's: the network now handles the variation that used to break it.

In [ ]:
# flatten images into vectors
X_train_flat = X_train.reshape(X_train.shape[0],-1)
X_test_flat  = X_test.reshape(X_test.shape[0],-1)

print("X_train_flat.shape:", X_train_flat.shape)
print("X_test_flat.shape: ", X_test_flat.shape)

In [ ]:
dropout_rate = 0.1

n_features = X_train_flat.shape[1]
n_classes  = np.unique(y_train).shape[0]

# Create model
model = Sequential([
    Input(shape=(n_features,)),
    
    Dense(32),
    BatchNormalization(),
    Activation('relu'),
    Dropout(dropout_rate),
    
    Dense(16),
    BatchNormalization(),
    Activation('relu'),
    Dropout(dropout_rate),
    
    Dense(8, activation='relu'),
    
    Dense(n_classes, activation='softmax'),
])

# Define the optimizer with a custom learning rate
optimizer = Adam(learning_rate=0.01)

# Compile model
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=25,          # Stop after 5 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

# Display model summary
model.summary()

In [ ]:
model, history = helpers.train_and_evaluate(
    model,
    X_train_flat, y_train,
    X_test_flat, y_test,
    epochs=30,
    batch_size=64,
    callbacks=[early_stopping]
)

### 5.3 What augmentation did — and didn't — fix

Training on the augmented images helps: the network finally sees digits at different shifts,
rotations, and scales, so it stops falling apart on them. But look at *what we had to do* — we
manually manufactured every shifted and rotated copy and forced the network to memorize them all.
We **brute-forced** the invariance.

That is the tell. A dense network has no built-in notion that a shifted digit is the same digit
(exactly what sections 3 and 4 showed), so the only way to make it robust to a transformation is to
show it that transformation explicitly, over and over. A convolutional network gets the same
robustness *for free* from its architecture — shared local filters and pooling — without ever being
shown an augmented image. Augmentation patches a hole in the model; convolution removes the hole.

## 6. The real test: CIFAR-10

MNIST let the dense network off easy. It hit ~98% — and even after we shuffled the pixels or nudged
the digit, it barely flinched, because a centered white digit on a black background is about the
simplest image a classifier will ever see.

**CIFAR-10** is what real images look like: 32×32 **color** photos of ten object classes —
airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck. The object can be anywhere in
the frame, at any scale, against cluttered natural backgrounds, in any color. Recognizing it means
detecting *local* features — edges, textures, parts — wherever they happen to appear. That is
exactly the kind of spatial structure sections 3 and 4 showed a dense network cannot exploit.

So let's give a genuine, well-sized dense network its best shot at CIFAR-10 and see how far it gets.

In [ ]:
from tensorflow.keras.datasets import cifar10

(Xc_train, yc_train), (Xc_test, yc_test) = cifar10.load_data()

cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

# Balanced sampling helper, reused from U2-2_CNN-6_TransferLearning: take an equal number of
# images per class so we train on a small subset instead of all 50,000.
def sample(data, labels, num_samples_per_digit):
    sampled_data = []
    sampled_labels = []
    for digit in range(10):
        digit_indices = np.where(labels == digit)[0]
        sampled_indices = np.random.choice(digit_indices, num_samples_per_digit, replace=False)
        sampled_data.append(data[sampled_indices])
        sampled_labels.append(labels[sampled_indices])
    return np.concatenate(sampled_data), np.concatenate(sampled_labels)

train_samples_per_class = 1000
test_samples_per_class  = 200

Xc_train, yc_train = sample(Xc_train, yc_train, train_samples_per_class)
Xc_test,  yc_test  = sample(Xc_test,  yc_test,  test_samples_per_class)

yc_train = yc_train.flatten()          # cifar labels come as (N, 1)
yc_test  = yc_test.flatten()

# One example per class -- these are what the network has to tell apart.
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for cls, ax in enumerate(axes.ravel()):
    ax.imshow(Xc_train[np.where(yc_train == cls)[0][0]])
    ax.set_title(cifar_classes[cls], fontsize=10); ax.axis('off')
plt.suptitle("CIFAR-10 — 32x32 colour photos of real objects"); plt.tight_layout(); plt.show()

print("Xc_train:", Xc_train.shape, "  Xc_test:", Xc_test.shape,
      f"  ({train_samples_per_class}/class train, {test_samples_per_class}/class test)")
print(f"Flattened feature length: {Xc_train[0].size} (vs 784 for MNIST)")

### 6.1 Same idea, a bigger dense network

To be fair — so nobody can say we crippled it — we give the dense network a **generous**
architecture (512 → 256 → 128 units, over a million parameters) and normalize the pixels to [0, 1]
so it trains cleanly. As in the transfer-learning notebook, we train on a small **balanced subset**
rather than all 50,000 images, so the cell runs quickly in class; the full set only buys the dense
net a few extra points and changes nothing about the verdict.

The one advantage it does *not* get is architectural: it is still fully connected, so it still sees
a flat bag of 3,072 numbers with no notion of which pixels are neighbors.

In [ ]:
# Normalise to [0,1] -- gives the dense net its best chance (can't blame preprocessing).
Xc_train_flat = (Xc_train.reshape(len(Xc_train), -1) / 255.0).astype('float32')
Xc_test_flat  = (Xc_test.reshape(len(Xc_test),  -1) / 255.0).astype('float32')

model_cifar = Sequential([
    Input(shape=(Xc_train_flat.shape[1],)),
    Dense(512), BatchNormalization(), Activation('relu'), Dropout(0.3),
    Dense(256), BatchNormalization(), Activation('relu'), Dropout(0.3),
    Dense(128), Activation('relu'),
    Dense(10, activation='softmax'),
])
model_cifar.compile(optimizer=Adam(learning_rate=1e-3),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stopping_cifar = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
model_cifar.summary()

In [ ]:
model_cifar, history_cifar = helpers.train_and_evaluate(
    model_cifar,
    Xc_train_flat, yc_train,
    Xc_test_flat, yc_test,
    epochs=30,
    batch_size=128,
    callbacks=[early_stopping_cifar],
    class_names=cifar_classes
)

In [ ]:
# helpers.train_and_evaluate already showed the loss curve, confusion matrix, and report.
# The one extra view worth having here is accuracy against the 10% chance line.
val_acc = (model_cifar.predict(Xc_test_flat, verbose=0).argmax(axis=1) == yc_test).mean()

plt.figure(figsize=(6, 4))
plt.plot(history_cifar.history['accuracy'],     label='train accuracy')
plt.plot(history_cifar.history['val_accuracy'], label='val accuracy')
plt.axhline(0.10, ls='--', color='gray', label='random guessing (10%)')
plt.title('CIFAR-10, dense network')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.ylim(0, 1); plt.legend()
plt.tight_layout(); plt.show()

print(f"Dense network on CIFAR-10 — test accuracy: {val_acc:.3f}")

### 6.2 The same data, a small CNN

For contrast, the same style of dense network scored about 0.98 on MNIST. On real photographs it
lands somewhere in the mid-40s — barely four times better than guessing.

Everything the dense network struggled with, a **convolutional** network is built to handle. Same
CIFAR-10 subset, same training budget — but now we keep each image as a 32×32×3 grid instead of
flattening it, and let convolutional layers scan for *local* patterns. Watch two numbers: the test
accuracy, and the parameter count.

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D

# The SAME CIFAR subset -- but kept as 32x32x3 images (never flattened) and normalised.
Xc_train_img = Xc_train.astype('float32') / 255.0
Xc_test_img  = Xc_test.astype('float32') / 255.0

model_cnn = Sequential([
    Input(shape=(32, 32, 3)),
    Conv2D(32, 3, padding='same', activation='linear'),
    Conv2D(32, 3, padding='same', activation='relu'),
    MaxPooling2D(),
    Conv2D(64, 3, padding='same', activation='linear'),
    Conv2D(64, 3, padding='same', activation='relu'),
    MaxPooling2D(),
    Conv2D(64, 3, padding='same', activation='relu'),
    GlobalAveragePooling2D(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax'),
])
model_cnn.compile(optimizer=Adam(learning_rate=1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_cnn.summary()

# Conv layers are heavier per epoch than the dense net, but converge fast -- 20 epochs with
# early stopping is plenty to make the point.
early_stopping_cnn = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model_cnn, history_cnn = helpers.train_and_evaluate(
    model_cnn,
    Xc_train_img, yc_train,
    Xc_test_img, yc_test,
    epochs=20,
    batch_size=128,
    callbacks=[early_stopping_cnn],
    class_names=cifar_classes
)

In [ ]:
cnn_val_acc  = (model_cnn.predict(Xc_test_img, verbose=0).argmax(axis=1) == yc_test).mean()
dense_params = model_cifar.count_params()
cnn_params   = model_cnn.count_params()

print(f"{'':10s}{'parameters':>14s}{'test accuracy':>16s}")
print(f"{'dense':10s}{dense_params:>14,}{val_acc:>16.3f}")
print(f"{'CNN':10s}{cnn_params:>14,}{cnn_val_acc:>16.3f}")
print(f"\nThe CNN uses ~{dense_params/cnn_params:.0f}x FEWER parameters and is far more accurate.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(['dense', 'CNN'], [val_acc, cnn_val_acc], color=['#4c72b0', '#dd8452'])
axes[0].axhline(0.10, ls='--', color='gray', label='chance (10%)')
axes[0].set_ylim(0, 1); axes[0].set_ylabel('test accuracy'); axes[0].set_title('Accuracy on CIFAR-10')
axes[0].legend()
for i, v in enumerate([val_acc, cnn_val_acc]):
    axes[0].text(i, v, f"{v:.2f}", ha='center', va='bottom')

axes[1].bar(['dense', 'CNN'], [dense_params, cnn_params], color=['#4c72b0', '#dd8452'])
axes[1].set_yscale('log'); axes[1].set_ylabel('parameters (log scale)'); axes[1].set_title('Model size')
for i, v in enumerate([dense_params, cnn_params]):
    axes[1].text(i, v, f"{v:,}", ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()

## 7. Review

| Experiment | What we did | What happened | What it proves |
|---|---|---|---|
| §3 Pixel shuffle | One fixed permutation of all 784 pixels, train and test | Accuracy essentially **unchanged** | The dense net never used adjacency, so destroying it costs nothing |
| §4 Translation | Compared a digit to a 3-pixel shift of itself | Low cosine similarity; **identical** after shuffling | A small shift is not "small" in the dense net's representation |
| §5 Augmentation | Trained on randomly transformed digits | Robustness **recovered** | The invariance can be bought — but only by memorizing every case |
| §6 CIFAR-10 | 1M-parameter dense net vs. a small CNN | Dense stalls in the mid-40s; CNN far higher with **fewer** parameters | Architecture, not capacity, is the binding constraint |

**Takeaways**

- **A dense layer treats every pixel as an independent input.** It has no way to know that
  neighboring pixels form edges and shapes (the shuffle demo), or that a patch of "fur" or "wheel"
  texture means the same thing wherever it appears (the translation demo).
- **A high MNIST score proves less than it looks.** A centered white digit on a black background is
  nearly a tabular problem in disguise. The shuffle experiment is the honest test, and the dense
  network passes it for exactly the wrong reason.
- **Augmentation patches a hole; convolution removes it.** Both give you robustness to shifts and
  rotations, but only one of them gets it from the architecture rather than from brute-force
  memorization.
- **More parameters do not fix a wrong prior.** Over a million dense parameters lose to a far
  smaller CNN on the same images. Convolution wins by building in three assumptions that happen to
  be true of images: locality (filters are small), weight sharing (the same filter runs everywhere),
  and tolerance to small shifts (pooling).

**Next:** `U2-2_CNN-2_MNIST.ipynb` builds the convolutional network this notebook has been arguing
for, and looks inside its feature maps.